In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
from src.utils.common import read_config, build_table_name, write_delta_table
from src.transformations.refined import (
    transform_refined_orders,
    transform_refined_customers,
    transform_refined_products
)
from src.utils.dqx_utils import apply_dq_checks

In [0]:
try:
    config = read_config("../config/dev_config.json")
    print("Config loaded successfully")

except Exception as error:
    raise RuntimeError(f"Failed to read config file: {error}")



In [0]:
try:
    raw_orders_table = build_table_name(config, "raw", "orders")
    raw_customers_table = build_table_name(config, "raw", "customers")
    raw_products_table = build_table_name(config, "raw", "products")

    refined_orders_table = build_table_name(config, "refined", "orders")
    refined_customers_table = build_table_name(config, "refined", "customers")
    refined_products_table = build_table_name(config, "refined", "products")

    print("Table names created successfully")

except Exception as error:
    raise RuntimeError(f"Failed to build table names: {error}")

In [0]:
try:
    raw_orders_df = spark.table(raw_orders_table)
    raw_customers_df = spark.table(raw_customers_table)
    raw_products_df = spark.table(raw_products_table)

    print("Raw tables read successfully")

except Exception as error:
    raise RuntimeError(f"Failed to read raw tables: {error}")

In [0]:
try:
    refined_orders_df = transform_refined_orders(raw_orders_df)
    refined_customers_df = transform_refined_customers(raw_customers_df)
    refined_products_df = transform_refined_products(raw_products_df)

    print("Refined transformations completed successfully")

except Exception as error:
    raise RuntimeError(f"Failed during refined transformations: {error}")

In [0]:
orders_dq_rules_path = "../config/dq_rules/orders_dq_rules.yml"
customers_dq_rules_path = "../config/dq_rules/customers_dq_rules.yml"
products_dq_rules_path = "../config/dq_rules/products_dq_rules.yml"

valid_orders_df, quarantine_orders_df = apply_dq_checks(
    refined_orders_df,
    orders_dq_rules_path
)

valid_customers_df, quarantine_customers_df = apply_dq_checks(
    refined_customers_df,
    customers_dq_rules_path
)

valid_products_df, quarantine_products_df = apply_dq_checks(
    refined_products_df,
    products_dq_rules_path
)

In [0]:
display(quarantine_products_df)

In [0]:
try:
    write_delta_table(valid_orders_df, refined_orders_table, mode="overwrite")
    write_delta_table(valid_customers_df, refined_customers_table, mode="overwrite")
    write_delta_table(valid_products_df, refined_products_table, mode="overwrite")

    write_delta_table(
        quarantine_orders_df,
        "dev.refined.quarantine_orders",
        mode="overwrite"
    )

    write_delta_table(
        quarantine_customers_df,
        "dev.refined.quarantine_customers",
        mode="overwrite"
    )

    write_delta_table(
        quarantine_products_df,
        "dev.refined.quarantine_products",
        mode="overwrite"
    )

    print("Refined and quarantine tables written successfully")

except Exception as error:
    raise RuntimeError(f"Failed to write refined/quarantine tables: {error}")

In [0]:
try:
    print("Orders")
    print("Raw count:", raw_orders_df.count())
    print("Refined count:", spark.table(refined_orders_table).count())

    print("Customers")
    print("Raw count:", raw_customers_df.count())
    print("Refined count:", spark.table(refined_customers_table).count())

    print("Products")
    print("Raw count:", raw_products_df.count())
    print("Refined count:", spark.table(refined_products_table).count())

except Exception as error:
    raise RuntimeError(f"Failed during count validation: {error}")